# Module 28 — Sampling Strategies

Every generation loop so far (Modules 17-18) called `torch.multinomial` on
the raw softmax distribution. That's one choice among several for turning
a model's output logits into an actual next token — this module covers
the standard family: **greedy**, **temperature**, **top-k**, and **top-p
(nucleus)** sampling, implemented from scratch and verified against their
defining properties.

## 1. A toy distribution to sample from

5 tokens with clearly different probabilities, so it's easy to see exactly
what each strategy keeps or discards.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)
logits = torch.tensor([2.5, 1.0, 0.5, 0.2, -1.0])
probs = F.softmax(logits, dim=-1)
for i, p in enumerate(probs.tolist()):
    print(f"token {i}: probability {p:.3f}")

## 2. Greedy: always the single most likely token

In [ ]:
def greedy(logits):
    return torch.argmax(logits).item()


chosen = [greedy(logits) for _ in range(10)]
assert all(c == chosen[0] for c in chosen)
print(f"greedy always picks token {chosen[0]} (highest probability): {chosen}")

## 3. Temperature: sharpening or flattening the distribution

Dividing logits by `T` before softmax: `T < 1` makes the distribution
*sharper* (more confident, closer to greedy); `T > 1` makes it *flatter*
(closer to uniform, more random). `T = 1` is the plain softmax.

In [ ]:
def sample_with_temperature(logits, temperature, generator=None):
    scaled_probs = F.softmax(logits / temperature, dim=-1)
    return torch.multinomial(scaled_probs, num_samples=1, generator=generator).item(), scaled_probs


gen = torch.Generator().manual_seed(0)

_, probs_low_temp = sample_with_temperature(logits, temperature=0.01, generator=gen)
_, probs_high_temp = sample_with_temperature(logits, temperature=5.0, generator=gen)

print("probs at T=0.01 (near-greedy):", [round(p, 3) for p in probs_low_temp.tolist()])
print("probs at T=5.0  (near-uniform):", [round(p, 3) for p in probs_high_temp.tolist()])

# As T -> 0, sampling should converge to greedy essentially always
samples_low_temp = [sample_with_temperature(logits, 0.01, gen)[0] for _ in range(200)]
greedy_token = greedy(logits)
match_rate = sum(s == greedy_token for s in samples_low_temp) / len(samples_low_temp)
print(f"\nAt T=0.01, fraction of 200 samples matching the greedy token: {match_rate:.1%}")
assert match_rate > 0.99

## 4. Top-k: only consider the k most likely tokens

Zero out every probability outside the top `k`, then renormalize what's
left so it still sums to 1. `k=1` should be mathematically identical to
greedy.

In [ ]:
def top_k_filter(logits, k):
    top_values, top_indices = torch.topk(logits, k)
    filtered = torch.full_like(logits, float("-inf"))
    filtered[top_indices] = top_values
    return filtered


def top_k_sample(logits, k, generator=None):
    filtered_logits = top_k_filter(logits, k)
    probs = F.softmax(filtered_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1, generator=generator).item(), probs


# k=1 must always match greedy
for _ in range(20):
    token, _ = top_k_sample(logits, k=1, generator=gen)
    assert token == greedy_token
print(f"top-k with k=1 always matches greedy (token {greedy_token}), confirmed over 20 draws.")

_, probs_k2 = top_k_sample(logits, k=2, generator=gen)
nonzero_tokens = (probs_k2 > 0).sum().item()
assert nonzero_tokens == 2
print(f"top-k with k=2 keeps exactly {nonzero_tokens} tokens with nonzero probability: {[round(p, 3) for p in probs_k2.tolist()]}")

## 5. Top-p (nucleus): keep the smallest set of tokens whose cumulative probability exceeds p

Unlike top-k's fixed count, top-p adapts: if the distribution is very
peaked, the "nucleus" might be just 1-2 tokens; if it's flat, it might need
many. The invariant to verify: the kept tokens' probability must sum to at
least `p` (and dropping the lowest-probability *kept* token would fall
below it).

In [ ]:
def top_p_filter(logits, p):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # keep the smallest prefix whose cumulative probability is >= p
    n_keep = (cumulative_probs < p).sum().item() + 1
    n_keep = min(n_keep, len(logits))

    keep_indices = sorted_indices[:n_keep]
    filtered = torch.full_like(logits, float("-inf"))
    filtered[keep_indices] = logits[keep_indices]
    return filtered, n_keep


def top_p_sample(logits, p, generator=None):
    filtered_logits, n_keep = top_p_filter(logits, p)
    probs = F.softmax(filtered_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1, generator=generator).item(), probs, n_keep


for p_threshold in [0.5, 0.9, 0.99]:
    _, probs_p, n_keep = top_p_sample(logits, p_threshold, gen)
    kept_prob_mass = probs_p[probs_p > 0].sum().item()
    original_prob_mass = probs[torch.topk(logits, n_keep).indices].sum().item()
    print(f"p={p_threshold}: kept {n_keep} tokens, original cumulative probability {original_prob_mass:.3f} (>= {p_threshold})")
    assert original_prob_mass >= p_threshold - 1e-6

## 6. Comparing diversity across strategies

In [ ]:
def unique_count(sample_fn, n=200):
    return len(set(sample_fn() for _ in range(n)))


gen2 = torch.Generator().manual_seed(1)
print("unique tokens seen in 200 draws:")
print("  greedy:            ", unique_count(lambda: greedy(logits)))
print("  temperature=0.01:  ", unique_count(lambda: sample_with_temperature(logits, 0.01, gen2)[0]))
print("  temperature=1.0:   ", unique_count(lambda: sample_with_temperature(logits, 1.0, gen2)[0]))
print("  top-k=2:           ", unique_count(lambda: top_k_sample(logits, 2, gen2)[0]))
print("  top-p=0.9:         ", unique_count(lambda: top_p_sample(logits, 0.9, gen2)[0]))

## Recap

- Greedy is deterministic; verified top-k with `k=1` is mathematically
  identical to it.
- Temperature reshapes the distribution's sharpness; verified that a very
  low temperature converges to greedy in practice (>99% match over 200
  samples).
- Top-p's kept token set always has cumulative probability at or above
  `p`, verified directly — unlike top-k, the number of tokens it keeps
  adapts to how peaked the distribution is.
- Measuring unique tokens sampled across 200 draws confirmed the intuitive
  ordering: greedy is least diverse, higher temperature and larger p are
  more diverse.

Module 29 covers KV-caching — making autoregressive generation with any of
these strategies fast, by not recomputing attention over the whole prefix
at every single new token.